#### Fine-Tuning BERT for POS Tagging & Chunking


In [1]:
from datasets import load_dataset

dataset = load_dataset("conll2003")

print(dataset)


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'default' at C:\Users\user\.cache\huggingface\datasets\conll2003\default\0.0.0\5a61f27da7e42956 (last modified on Sun Apr  5 19:12:22 2026).


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 219554
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 55044
    })
    test: Dataset({
        features: ['text'],
        num_rows: 50350
    })
})


### Dataset Selection

In [ ]:
from datasets import load_dataset

# Load CoNLL-2003 dataset
dataset = load_dataset("conll2003")

# Inspect labels
label_list = dataset["train"].features["ner_tags"].feature.names
pos_labels = dataset["train"].features["pos_tags"].feature.names
chunk_labels = dataset["train"].features["chunk_tags"].feature.names

print("POS Labels:", pos_labels)
print("Chunk Labels:", chunk_labels)


### Data Preprocessing

In [9]:
def tokenize_and_align_labels(examples):
    # 'tokens' is the list of words for each sentence
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        truncation=True, 
        is_split_into_words=True
    )

    labels = []
    # 'chunk_tags' contains the numeric IDs for chunk labels
    for i, label in enumerate(examples["chunk_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100) # Special tokens (CLS, SEP)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx]) # First subword gets tag
            else:
                label_ids.append(-100) # Other subwords ignored
            previous_word_idx = word_idx
        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

###  Model Setup

In [ ]:
from transformers import AutoModelForTokenClassification

num_labels = len(pos_labels)

model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels,
    id2label={i: l for i, l in enumerate(pos_labels)},
    label2id={l: i for i, l in enumerate(pos_labels)}
)


### Training

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer
)

trainer.train()


###  Evaluation 

In [ ]:
import numpy as np
from datasets import load_metric

metric = load_metric("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [pos_labels[p] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    true_labels = [
        [pos_labels[l] for (p, l) in zip(pred, lab) if l != -100]
        for pred, lab in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
    }

trainer.compute_metrics = compute_metrics

trainer.evaluate()


In [ ]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    predictions = outputs.argmax(dim=2)

    predicted_labels = [pos_labels[p.item()] for p in predictions[0]]

    return list(zip(tokens, predicted_labels))


sentence = "John works at Google in California"
print(predict(sentence))


In [5]:
comparison = """
POS Tagging:
- Assigns grammatical roles (noun, verb, adjective)
- Word-level classification
- Easier task

Chunking:
- Groups words into phrases (NP, VP)
- Phrase-level classification
- More complex than POS

Conclusion:
Chunking requires understanding structure beyond individual words.
"""
print(comparison)



POS Tagging:
- Assigns grammatical roles (noun, verb, adjective)
- Word-level classification
- Easier task

Chunking:
- Groups words into phrases (NP, VP)
- Phrase-level classification
- More complex than POS

Conclusion:
Chunking requires understanding structure beyond individual words.



In [6]:
report = """
Challenges:
- Handling subword tokenization
- Aligning labels with tokens
- Ignoring special tokens (-100)

Observations:
- DistilBERT performs well on POS tagging
- Chunking requires more contextual understanding

Insights:
- Transformer models excel in sequence labeling tasks
- Proper preprocessing is critical
"""
print(report)



Challenges:
- Handling subword tokenization
- Aligning labels with tokens
- Ignoring special tokens (-100)

Observations:
- DistilBERT performs well on POS tagging
- Chunking requires more contextual understanding

Insights:
- Transformer models excel in sequence labeling tasks
- Proper preprocessing is critical

